# 1.3 知识蒸馏（Knowledge Distillation）

知识蒸馏将大模型（教师模型）的知识迁移到小模型（学生模型），使小模型在参数量大幅减少的情况下仍能逼近大模型的性能。根据是否可访问教师模型内部状态，分为白盒蒸馏和黑盒蒸馏。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
## 1.3.1 白盒蒸馏（White-Box Distillation）

可访问教师模型的内部状态（logits、中间层特征、注意力图等），利用这些丰富信息指导学生模型训练。

### Logits级蒸馏（Response-Based）

学生模型的输出logits与教师模型的soft logits之间计算KL散度损失。教师输出的概率分布包含"暗知识"（类别间的相似性信息），比硬标签信息更丰富。

$$L_{KD} = \alpha \cdot T^2 \cdot KL(\sigma(z_s/T) || \sigma(z_t/T)) + (1-\alpha) \cdot L_{CE}(y_s, y_{true})$$

其中 $T$ 是温度参数，$T$ 越大soft label越平滑，暗知识越丰富。

In [ ]:
class TeacherModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 32), nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        return self.net(x)

def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    """知识蒸馏损失 = α·T²·KL散度 + (1-α)·交叉熵"""
    soft_teacher = F.log_softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    kl_loss = F.kl_div(soft_student, soft_teacher.exp(), reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits, labels)
    return alpha * kl_loss + (1 - alpha) * ce_loss

def train_with_distillation(teacher, student, train_loader, epochs=50,
                            temperature=4.0, alpha=0.7, lr=1e-3):
    """使用知识蒸馏训练学生模型"""
    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            with torch.no_grad():
                teacher_logits = teacher(x)
            student_logits = student(x)
            loss = distillation_loss(student_logits, teacher_logits, y,
                                     temperature=temperature, alpha=alpha)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

def train_normal(model, train_loader, epochs=50, lr=1e-3):
    """普通训练（无蒸馏）"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            output = model(x)
            loss = F.cross_entropy(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

dim, n_classes = 64, 10
x_data = torch.randn(512, dim)
y_data = torch.randint(0, n_classes, (512,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=64)

teacher = TeacherModel(dim, n_classes)
train_normal(teacher, loader, epochs=30)

student_kd = StudentModel(dim, n_classes)
student_no_kd = StudentModel(dim, n_classes)

loss_kd = train_with_distillation(teacher, student_kd, loader, epochs=30, temperature=4.0)
loss_no_kd = train_normal(student_no_kd, loader, epochs=30)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student_kd.parameters())

with torch.no_grad():
    acc_teacher = (teacher(x_data).argmax(1) == y_data).float().mean()
    acc_kd = (student_kd(x_data).argmax(1) == y_data).float().mean()
    acc_no_kd = (student_no_kd(x_data).argmax(1) == y_data).float().mean()

print(f"=== Logits级蒸馏效果 ===")
print(f"教师模型参数量: {teacher_params:,}, 准确率: {acc_teacher:.4f}")
print(f"学生模型参数量: {student_params:,} ({student_params/teacher_params:.1%} of teacher)")
print(f"学生(有蒸馏)准确率: {acc_kd:.4f}")
print(f"学生(无蒸馏)准确率: {acc_no_kd:.4f}")
print(f"蒸馏提升: {(acc_kd - acc_no_kd)*100:.2f}%")

print(f"\n=== 温度参数影响 ===")
for T in [1.0, 2.0, 4.0, 8.0, 16.0]:
    s = StudentModel(dim, n_classes)
    train_with_distillation(teacher, s, loader, epochs=30, temperature=T)
    with torch.no_grad():
        acc = (s(x_data).argmax(1) == y_data).float().mean()
    print(f"T={T:<5} 准确率: {acc:.4f}")

### 特征级蒸馏（Feature-Level / Intermediate-Layer）

对齐教师和学生中间层的隐藏状态或特征表示，通常需要线性变换层将学生的特征映射到教师的特征空间。

In [ ]:
class TeacherWithFeatures(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(dim, 256), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(256, 128), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Linear(128, 64), nn.ReLU())
        self.head = nn.Linear(64, n_classes)

    def forward(self, x, return_features=False):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        out = self.head(f3)
        if return_features:
            return out, [f1, f2, f3]
        return out

class StudentWithFeatures(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(dim, 64), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Linear(32, 16), nn.ReLU())
        self.head = nn.Linear(16, n_classes)
        self.align1 = nn.Linear(64, 256)
        self.align2 = nn.Linear(32, 128)
        self.align3 = nn.Linear(16, 64)

    def forward(self, x, return_features=False):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        out = self.head(f3)
        if return_features:
            aligned_f1 = self.align1(f1)
            aligned_f2 = self.align2(f2)
            aligned_f3 = self.align3(f3)
            return out, [aligned_f1, aligned_f2, aligned_f3]
        return out

def feature_distillation_loss(student_out, teacher_out, student_features, teacher_features,
                              labels, temperature=4.0, alpha=0.5, beta=0.3):
    """特征级蒸馏损失 = α·logits蒸馏 + β·特征蒸馏 + (1-α-β)·CE"""
    soft_t = F.log_softmax(teacher_out / temperature, dim=-1)
    soft_s = F.log_softmax(student_out / temperature, dim=-1)
    kl_loss = F.kl_div(soft_s, soft_t.exp(), reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_out, labels)

    feat_loss = 0
    for sf, tf in zip(student_features, teacher_features):
        feat_loss += F.mse_loss(sf, tf.detach())
    feat_loss /= len(student_features)

    return alpha * kl_loss + beta * feat_loss + (1 - alpha - beta) * ce_loss

teacher_feat = TeacherWithFeatures(dim, n_classes)
train_normal(teacher_feat, loader, epochs=30)

student_feat = StudentWithFeatures(dim, n_classes)
teacher_feat.eval()
optimizer = torch.optim.Adam(student_feat.parameters(), lr=1e-3)

for epoch in range(30):
    for x, y in loader:
        with torch.no_grad():
            t_out, t_feats = teacher_feat(x, return_features=True)
        s_out, s_feats = student_feat(x, return_features=True)
        loss = feature_distillation_loss(s_out, t_out, s_feats, t_feats, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

student_logits_only = StudentModel(dim, n_classes)
train_with_distillation(teacher, student_logits_only, loader, epochs=30)

with torch.no_grad():
    acc_feat = (student_feat(x_data).argmax(1) == y_data).float().mean()
    acc_logits = (student_logits_only(x_data).argmax(1) == y_data).float().mean()

print(f"=== 特征级蒸馏 vs Logits级蒸馏 ===")
print(f"Logits级蒸馏准确率: {acc_logits:.4f}")
print(f"特征级蒸馏准确率: {acc_feat:.4f}")
print(f"\n特征级蒸馏优势: 利用中间层信息，提供更丰富的梯度信号")
print(f"代价: 需要额外的对齐层，训练更复杂")

### 注意力蒸馏（Attention Transfer）

让学生模型的注意力图模仿教师模型的注意力分布，学习教师模型的注意力模式。

In [ ]:
class AttentionTeacher(nn.Module):
    def __init__(self, dim=64, n_heads=4, n_classes=10):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.head = nn.Linear(dim, n_classes)
        self.n_heads = n_heads

    def forward(self, x, return_attn=False):
        B, N, C = x.shape
        qkv = F.linear(x, self.attn.in_proj_weight, self.attn.in_proj_bias)
        q, k, v = qkv.chunk(3, dim=-1)
        head_dim = C // self.n_heads
        q = q.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        k = k.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        v = v.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn_probs = F.softmax(attn_weights, dim=-1)
        out = torch.matmul(attn_probs, v)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        out = self.attn.out_proj(out)
        logits = self.head(out.mean(dim=1))
        if return_attn:
            return logits, attn_probs
        return logits

class AttentionStudent(nn.Module):
    def __init__(self, dim=64, n_heads=2, n_classes=10):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.head = nn.Linear(dim, n_classes)
        self.n_heads = n_heads

    def forward(self, x, return_attn=False):
        B, N, C = x.shape
        qkv = F.linear(x, self.attn.in_proj_weight, self.attn.in_proj_bias)
        q, k, v = qkv.chunk(3, dim=-1)
        head_dim = C // self.n_heads
        q = q.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        k = k.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        v = v.view(B, N, self.n_heads, head_dim).transpose(1, 2)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn_probs = F.softmax(attn_weights, dim=-1)
        out = torch.matmul(attn_probs, v)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        out = self.attn.out_proj(out)
        logits = self.head(out.mean(dim=1))
        if return_attn:
            return logits, attn_probs
        return logits

def attention_distillation_loss(s_attn, t_attn, student_logits, labels, gamma=0.1):
    """注意力蒸馏损失"""
    s_attn_sum = s_attn.sum(dim=1, keepdim=True)
    t_attn_sum = t_attn.sum(dim=1, keepdim=True)
    n_s, n_t = s_attn.size(1), t_attn.size(1)
    if n_s != n_t:
        t_attn_pooled = t_attn_sum.expand(-1, n_s, -1, -1) / n_s
        s_attn_pooled = s_attn
    else:
        t_attn_pooled = t_attn
        s_attn_pooled = s_attn
    min_len = min(s_attn_pooled.size(-1), t_attn_pooled.size(-1))
    s_attn_crop = s_attn_pooled[:, :, :min_len, :min_len]
    t_attn_crop = t_attn_pooled[:, :, :min_len, :min_len]
    attn_loss = F.mse_loss(s_attn_crop, t_attn_crop.detach())
    ce_loss = F.cross_entropy(student_logits, labels)
    return ce_loss + gamma * attn_loss

x_seq = torch.randn(64, 8, 64)
y_seq = torch.randint(0, 10, (64,))
seq_dataset = torch.utils.data.TensorDataset(x_seq, y_seq)
seq_loader = torch.utils.data.DataLoader(seq_dataset, batch_size=16)

attn_teacher = AttentionTeacher(dim=64, n_heads=4, n_classes=10)
attn_student = AttentionStudent(dim=64, n_heads=2, n_classes=10)

optimizer_t = torch.optim.Adam(attn_teacher.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in seq_loader:
        loss = F.cross_entropy(attn_teacher(x), y)
        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

attn_teacher.eval()
optimizer_s = torch.optim.Adam(attn_student.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in seq_loader:
        with torch.no_grad():
            _, t_attn = attn_teacher(x, return_attn=True)
        s_logits, s_attn = attn_student(x, return_attn=True)
        loss = attention_distillation_loss(s_attn, t_attn, s_logits, y, gamma=0.1)
        optimizer_s.zero_grad()
        loss.backward()
        optimizer_s.step()

with torch.no_grad():
    acc_attn = (attn_student(x_seq).argmax(1) == y_seq).float().mean()
    acc_teacher = (attn_teacher(x_seq).argmax(1) == y_seq).float().mean()

print(f"=== 注意力蒸馏效果 ===")
print(f"教师(4头)准确率: {acc_teacher:.4f}")
print(f"学生(2头+注意力蒸馏)准确率: {acc_attn:.4f}")
print(f"\n注意力蒸馏让学生学会教师的注意力模式，提升小模型性能")

---
## 1.3.2 黑盒蒸馏（Black-Box Distillation）

仅能通过API访问教师模型的输出（生成的文本），无法获取内部状态。通过教师模型生成的高质量数据来训练学生模型。

### 指令蒸馏（Instruction Distillation）

使用教师模型对大量指令生成回答，构建指令微调数据集训练学生模型。如Alpaca、Vicuna等方法。

In [ ]:
class SimpleLLM(nn.Module):
    """简化版语言模型，用于演示指令蒸馏"""
    def __init__(self, vocab_size=1000, dim=64, n_classes=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=4, batch_first=True),
            num_layers=2
        )
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        h = self.emb(x)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))

vocab_size = 1000
teacher_llm = SimpleLLM(vocab_size, dim=64, n_classes=10)
student_llm = SimpleLLM(vocab_size, dim=32, n_classes=10)

optimizer_t = torch.optim.Adam(teacher_llm.parameters(), lr=1e-3)
x_tokens = torch.randint(0, vocab_size, (256, 16))
y_labels = torch.randint(0, 10, (256,))
token_dataset = torch.utils.data.TensorDataset(x_tokens, y_labels)
token_loader = torch.utils.data.DataLoader(token_dataset, batch_size=32)

for _ in range(20):
    for x, y in token_loader:
        loss = F.cross_entropy(teacher_llm(x), y)
        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

print(f"=== 黑盒指令蒸馏 ===")
print(f"步骤1: 使用教师模型生成标注数据")

n_synthetic = 512
x_synthetic = torch.randint(0, vocab_size, (n_synthetic, 16))
with torch.no_grad():
    y_synthetic = teacher_llm(x_synthetic).argmax(dim=1)

print(f"生成合成数据: {n_synthetic}条")
print(f"合成标签分布: {torch.bincount(y_synthetic, minlength=10)}")

syn_dataset = torch.utils.data.TensorDataset(x_synthetic, y_synthetic)
syn_loader = torch.utils.data.DataLoader(syn_dataset, batch_size=32)

print(f"\n步骤2: 使用合成数据训练学生模型")
optimizer_s = torch.optim.Adam(student_llm.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in syn_loader:
        loss = F.cross_entropy(student_llm(x), y)
        optimizer_s.zero_grad()
        loss.backward()
        optimizer_s.step()

student_no_distill = SimpleLLM(vocab_size, dim=32, n_classes=10)
optimizer_n = torch.optim.Adam(student_no_distill.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in token_loader:
        loss = F.cross_entropy(student_no_distill(x), y)
        optimizer_n.zero_grad()
        loss.backward()
        optimizer_n.step()

with torch.no_grad():
    acc_teacher = (teacher_llm(x_tokens).argmax(1) == y_labels).float().mean()
    acc_student_kd = (student_llm(x_tokens).argmax(1) == y_labels).float().mean()
    acc_student_no = (student_no_distill(x_tokens).argmax(1) == y_labels).float().mean()

teacher_p = sum(p.numel() for p in teacher_llm.parameters())
student_p = sum(p.numel() for p in student_llm.parameters())

print(f"\n=== 黑盒蒸馏结果 ===")
print(f"教师模型: {teacher_p:,}参数, 准确率={acc_teacher:.4f}")
print(f"学生(黑盒蒸馏): {student_p:,}参数, 准确率={acc_student_kd:.4f}")
print(f"学生(无蒸馏): {student_p:,}参数, 准确率={acc_student_no:.4f}")
print(f"\n黑盒蒸馏关键: 合成数据的质量和多样性决定蒸馏效果")

### 自蒸馏（Self-Distillation）

模型自身作为教师，通过不同训练阶段或不同数据增强下的输出一致性来提升性能。

In [ ]:
class SelfDistillationTrainer:
    """自蒸馏训练器：使用EMA模型作为教师"""
    def __init__(self, model, ema_decay=0.99, temperature=4.0, alpha=0.5):
        self.model = model
        self.ema_decay = ema_decay
        self.temperature = temperature
        self.alpha = alpha
        self.ema_model = copy.deepcopy(model)
        for p in self.ema_model.parameters():
            p.requires_grad = False

    def update_ema(self):
        with torch.no_grad():
            for p_ema, p_model in zip(self.ema_model.parameters(), self.model.parameters()):
                p_ema.data = self.ema_decay * p_ema.data + (1 - self.ema_decay) * p_model.data

    def train_step(self, x, y):
        student_logits = self.model(x)
        with torch.no_grad():
            teacher_logits = self.ema_model(x)

        soft_t = F.log_softmax(teacher_logits / self.temperature, dim=-1)
        soft_s = F.log_softmax(student_logits / self.temperature, dim=-1)
        kl_loss = F.kl_div(soft_s, soft_t.exp(), reduction='batchmean') * (self.temperature ** 2)
        ce_loss = F.cross_entropy(student_logits, y)
        loss = self.alpha * kl_loss + (1 - self.alpha) * ce_loss
        return loss

model_self = StudentModel(dim=64, n_classes=10)
trainer = SelfDistillationTrainer(model_self, ema_decay=0.99, temperature=4.0, alpha=0.5)
optimizer = torch.optim.Adam(model_self.parameters(), lr=1e-3)

for epoch in range(30):
    for x, y in loader:
        loss = trainer.train_step(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trainer.update_ema()

model_baseline = StudentModel(dim=64, n_classes=10)
train_normal(model_baseline, loader, epochs=30)

with torch.no_grad():
    acc_self = (model_self(x_data).argmax(1) == y_data).float().mean()
    acc_base = (model_baseline(x_data).argmax(1) == y_data).float().mean()

print(f"=== 自蒸馏效果 ===")
print(f"普通训练准确率: {acc_base:.4f}")
print(f"自蒸馏(EMA教师)准确率: {acc_self:.4f}")
print(f"提升: {(acc_self - acc_base)*100:.2f}%")
print(f"\n自蒸馏优势: 无需额外教师模型，EMA提供更稳定的训练目标")
print(f"适用场景: 无法获取更大教师模型时的替代方案")

### 蒸馏方法综合对比

In [ ]:
print(f"{'方法':<25} {'需要教师内部状态':<20} {'训练成本':<15} {'精度上限':<15}")
print("-" * 75)
methods = [
    ("Logits蒸馏", "需要logits", "中等", "高"),
    ("特征级蒸馏", "需要中间层特征", "较高", "更高"),
    ("注意力蒸馏", "需要注意力图", "较高", "高"),
    ("指令蒸馏(黑盒)", "仅需输出文本", "低", "中等"),
    ("自蒸馏(EMA)", "无需教师", "低", "中等"),
    ("MiniLLM(反向KL)", "需要logits", "中等", "高(生成式)"),
]
for name, state, cost, acc in methods:
    print(f"{name:<25} {state:<20} {cost:<15} {acc:<15}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. 有教师模型访问权限: 白盒蒸馏(Logits+特征)效果最好")
print(f"2. 仅有API访问: 黑盒指令蒸馏，数据质量是关键")
print(f"3. 无教师模型: 自蒸馏(EMA)或渐进式训练")
print(f"4. 生成式LLM: MiniLLM的反向KL散度更适合")